# 📚 LangChain Methods — Complete Notebook

> **LangChain** is a framework for developing applications powered by large language models (LLMs).  
> This notebook covers core LangChain modules with code examples, explanations, and best practices.

---

## 📋 Table of Contents
1. [Installation & Setup](#1-installation--setup)
2. [LLMs & Chat Models](#2-llms--chat-models)
3. [Prompt Templates](#3-prompt-templates)
4. [Chains](#4-chains)
5. [Memory](#5-memory)
6. [Agents & Tools](#6-agents--tools)
7. [Document Loaders](#7-document-loaders)
8. [Text Splitters](#8-text-splitters)
9. [Embeddings & Vector Stores](#9-embeddings--vector-stores)
10. [Retrieval & RAG](#10-retrieval--rag)
11. [Output Parsers](#11-output-parsers)
12. [Callbacks & Streaming](#12-callbacks--streaming)
13. [LCEL — LangChain Expression Language](#13-lcel--langchain-expression-language)
14. [Structured Output & Tool Binding](#14-structured-output--tool-binding)
15. [Advanced Retrievers](#15-advanced-retrievers)
16. [Summarization Chains](#16-summarization-chains)
17. [Caching](#17-caching)
18. [Async Support](#18-async-support)
19. [Multi-modal (Vision)](#19-multi-modal-vision)
20. [LangSmith — Tracing & Debugging](#20-langsmith--tracing--debugging)
21. [LangGraph — Stateful Agent Workflows](#21-langgraph--stateful-agent-workflows)

---
## 1. Installation & Setup

In [ ]:
# Install core LangChain packages
!pip install langchain langchain-openai langchain-community langchain-core
!pip install openai tiktoken faiss-cpu chromadb pypdf python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from a .env file
load_dotenv()

# Set your OpenAI API key (or use .env file)
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

print("Environment configured ✅")

---
## 2. LLMs & Chat Models

LangChain provides two main interfaces for language models:
- **LLM**: Text-in, text-out (legacy)
- **ChatModel**: Message-in, message-out (modern & preferred)

In [ ]:
from langchain_openai import OpenAI, ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# ── Legacy LLM ──────────────────────────────────────────────
llm = OpenAI(model="gpt-3.5-turbo-instruct", temperature=0.7)
response = llm.invoke("What is LangChain in one sentence?")
print("LLM response:", response)

In [ ]:
# ── Chat Model ───────────────────────────────────────────────
chat = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = [
    SystemMessage(content="You are a helpful Python tutor."),
    HumanMessage(content="Explain decorators in Python briefly.")
]

response = chat.invoke(messages)
print("Role:", response.response_metadata)
print("Content:", response.content)

In [ ]:
# ── Key ChatOpenAI Parameters ────────────────────────────────
chat_configured = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,        # 0 = deterministic, 1 = creative
    max_tokens=512,         # max tokens in response
    timeout=30,             # request timeout in seconds
    max_retries=2,          # retry on failure
    streaming=False,        # enable streaming
)

# Batch invocations
responses = chat_configured.batch([
    [HumanMessage(content="What is 2+2?")],
    [HumanMessage(content="Capital of France?")],
])

for r in responses:
    print(r.content)

---
## 3. Prompt Templates

Prompt Templates allow you to create reusable, parameterized prompts.

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate

# ── Basic PromptTemplate ─────────────────────────────────────
template = PromptTemplate(
    input_variables=["topic", "level"],
    template="Explain {topic} in simple terms suitable for a {level} student."
)

prompt = template.format(topic="recursion", level="beginner")
print(prompt)

In [ ]:
# ── ChatPromptTemplate ───────────────────────────────────────
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Be concise and precise."),
    ("human", "Question: {question}")
])

formatted = chat_prompt.format_messages(
    domain="machine learning",
    question="What is gradient descent?"
)

for msg in formatted:
    print(f"[{msg.type.upper()}]: {msg.content}")

In [ ]:
# ── FewShotPromptTemplate ────────────────────────────────────
examples = [
    {"word": "happy", "antonym": "sad"},
    {"word": "tall",  "antonym": "short"},
    {"word": "fast",  "antonym": "slow"},
]

example_template = PromptTemplate(
    input_variables=["word", "antonym"],
    template="Word: {word} → Antonym: {antonym}"
)

few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="Give the antonym of every input:",
    suffix="Word: {input} → Antonym:",
    input_variables=["input"]
)

print(few_shot.format(input="bright"))

---
## 4. Chains

Chains link components together into a pipeline.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── LCEL Chain (recommended modern approach) ─────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template(
    "Give me 3 interesting facts about {country}."
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"country": "Japan"})
print(result)

In [ ]:
# ── Sequential Chain using LCEL ──────────────────────────────
from langchain_core.runnables import RunnablePassthrough

# Step 1: Generate a topic summary
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the topic '{topic}' in 2 sentences."
)
summary_chain = summary_prompt | llm | StrOutputParser()

# Step 2: Turn summary into a tweet
tweet_prompt = ChatPromptTemplate.from_template(
    "Turn this summary into a Twitter/X post under 280 chars:\n{summary}"
)
tweet_chain = tweet_prompt | llm | StrOutputParser()

# Combine: pass original topic, then pipe summary to tweet
full_chain = (
    {"summary": summary_chain, "topic": RunnablePassthrough()}
    | tweet_chain
)

result = full_chain.invoke({"topic": "quantum computing"})
print(result)

In [ ]:
# ── LLMChain (legacy, still widely used) ─────────────────────
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["product"],
    template="Write a catchy product tagline for: {product}"
)

chain = LLMChain(llm=llm, prompt=prompt, verbose=True)
result = chain.invoke({"product": "smart water bottle"})
print(result["text"])

---
## 5. Memory

Memory allows chains and agents to retain information across interactions.

In [ ]:
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory
)
from langchain.chains import ConversationChain

# ── ConversationBufferMemory: stores full history ────────────
memory = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory, verbose=False)

print(conversation.predict(input="Hi! My name is Alice."))
print(conversation.predict(input="What is my name?"))  # Remembers 'Alice'
print("\nMemory buffer:")
print(memory.buffer)

In [ ]:
# ── ConversationBufferWindowMemory: last k messages ──────────
window_memory = ConversationBufferWindowMemory(k=3)  # keep last 3 exchanges

conv_window = ConversationChain(llm=llm, memory=window_memory)

for i, msg in enumerate(["Hello", "Tell me about AI", "What is ML?", "And DL?"]):
    resp = conv_window.predict(input=msg)
    print(f"Turn {i+1}: {resp[:80]}...\n")

In [ ]:
# ── ConversationSummaryMemory: summarizes old messages ───────
summary_memory = ConversationSummaryMemory(llm=llm)

conv_summary = ConversationChain(llm=llm, memory=summary_memory)
conv_summary.predict(input="I work as a data scientist at a startup.")
conv_summary.predict(input="We mostly use Python and PyTorch.")

print("Summary so far:")
print(summary_memory.buffer)

In [ ]:
# ── Modern LCEL approach with message history ─────────────────
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}  # session_id → history

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly assistant."),
    ("placeholder", "{history}"),
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

config = {"configurable": {"session_id": "session_abc"}}

r1 = with_history.invoke({"input": "My favorite color is blue."}, config=config)
r2 = with_history.invoke({"input": "What is my favorite color?"}, config=config)
print(r2)

---
## 6. Agents & Tools

Agents use an LLM to reason and decide which tools to use dynamically.

In [ ]:
from langchain.tools import tool
from langchain_core.tools import Tool
import math

# ── Define custom tools with @tool decorator ─────────────────
@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression. Input: math expression as a string."""
    try:
        result = eval(expression, {"__builtins__": {}}, vars(math))
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def word_length(word: str) -> str:
    """Returns the number of characters in a word."""
    return str(len(word))

print(calculator.invoke({"expression": "sqrt(144) + 5"}))
print(word_length.invoke({"word": "LangChain"}))

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

tools = [calculator, word_length]

# Pull a standard ReAct prompt from LangChain Hub
react_prompt = hub.pull("hwchase17/react")

agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

result = agent_executor.invoke({"input": "What is sqrt(256) and how many letters in 'artificial'?"})
print("\nFinal Answer:", result["output"])

In [ ]:
# ── OpenAI Tools Agent (function-calling based) ───────────────
from langchain.agents import create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when needed."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

tools_agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=agent_prompt)

executor = AgentExecutor(agent=tools_agent, tools=tools, verbose=True)
result = executor.invoke({"input": "Compute 3 * (7 + 12) and find the length of 'intelligence'"})
print(result["output"])

---
## 7. Document Loaders

Load documents from various sources: files, URLs, databases, APIs.

In [ ]:
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    CSVLoader,
    WebBaseLoader,
    DirectoryLoader
)

# ── Text Loader ──────────────────────────────────────────────
# Creates a sample text file to load
with open("/tmp/sample.txt", "w") as f:
    f.write("LangChain is a powerful framework.\nIt supports many integrations.\n")

text_loader = TextLoader("/tmp/sample.txt")
text_docs = text_loader.load()
print("Text docs:", len(text_docs), "document(s)")
print("Content:", text_docs[0].page_content)
print("Metadata:", text_docs[0].metadata)

In [ ]:
# ── PDF Loader ────────────────────────────────────────────────
# pdf_loader = PyPDFLoader("/path/to/your/file.pdf")
# pdf_docs = pdf_loader.load()   # Returns one Document per page
# print(f"PDF pages loaded: {len(pdf_docs)}")
# print(pdf_docs[0].page_content[:300])
print("Uncomment above to load a real PDF file.")

In [ ]:
# ── Web Loader ────────────────────────────────────────────────
web_loader = WebBaseLoader("https://python.langchain.com/docs/introduction")
web_docs = web_loader.load()
print(f"Web documents loaded: {len(web_docs)}")
print("First 300 chars:", web_docs[0].page_content[:300])

In [ ]:
# ── CSV Loader ────────────────────────────────────────────────
import csv

with open("/tmp/sample.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "age", "city"])
    writer.writerows([["Alice", 30, "NYC"], ["Bob", 25, "LA"], ["Carol", 35, "Chicago"]])

csv_loader = CSVLoader("/tmp/sample.csv")
csv_docs = csv_loader.load()
print(f"CSV rows as documents: {len(csv_docs)}")
print(csv_docs[0].page_content)

---
## 8. Text Splitters

Split large documents into smaller chunks for embedding and retrieval.

In [ ]:
from langchain.text_splitter import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

sample_text = """
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
Machine Learning (ML) is a subset of AI that enables systems to learn from data.
Deep Learning (DL) is a subset of ML that uses neural networks with many layers.
Natural Language Processing (NLP) focuses on the interaction between computers and human language.
Large Language Models (LLMs) are a type of AI trained on vast amounts of text data.
LangChain is a framework that simplifies building applications powered by LLMs.
"""

# ── RecursiveCharacterTextSplitter (recommended) ─────────────
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = recursive_splitter.split_text(sample_text)
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)

In [ ]:
# ── Split Documents (not just text) ─────────────────────────
from langchain_core.documents import Document

docs = [
    Document(page_content=sample_text, metadata={"source": "ai_overview", "page": 1})
]

split_docs = recursive_splitter.split_documents(docs)
print(f"Split into {len(split_docs)} document chunks")
print("Metadata preserved:", split_docs[0].metadata)

---
## 9. Embeddings & Vector Stores

Convert text to vector embeddings and store them for semantic search.

In [ ]:
from langchain_openai import OpenAIEmbeddings

# ── Create Embeddings ────────────────────────────────────────
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Embed a single query
vector = embeddings.embed_query("What is machine learning?")
print(f"Embedding dimensions: {len(vector)}")
print(f"First 5 values: {vector[:5]}")

# Embed multiple documents
texts = ["Python is a programming language.", "LangChain builds LLM apps.", "Dogs love to play fetch."]
doc_vectors = embeddings.embed_documents(texts)
print(f"\nEmbedded {len(doc_vectors)} documents")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# ── FAISS Vector Store ───────────────────────────────────────
docs = [
    Document(page_content="LangChain simplifies building LLM applications.", metadata={"topic": "langchain"}),
    Document(page_content="Python is widely used in data science and AI.", metadata={"topic": "python"}),
    Document(page_content="Vector databases store embeddings for similarity search.", metadata={"topic": "vectordb"}),
    Document(page_content="GPT-4 is a large language model by OpenAI.", metadata={"topic": "llm"}),
    Document(page_content="Retrieval Augmented Generation improves LLM responses.", metadata={"topic": "rag"}),
]

# Build FAISS index from documents
vectorstore = FAISS.from_documents(docs, embeddings)

# Similarity search
query = "How do vector stores work?"
results = vectorstore.similarity_search(query, k=2)

print(f"Query: '{query}'")
print("\nTop results:")
for i, doc in enumerate(results):
    print(f"  {i+1}. [{doc.metadata['topic']}] {doc.page_content}")

In [ ]:
# ── Similarity search with scores ────────────────────────────
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print("Results with similarity scores (lower = more similar for FAISS L2):")
for doc, score in results_with_scores:
    print(f"  Score: {score:.4f} | {doc.page_content}")

# ── Save and load ─────────────────────────────────────────────
vectorstore.save_local("/tmp/faiss_index")
loaded_vs = FAISS.load_local("/tmp/faiss_index", embeddings, allow_dangerous_deserialization=True)
print("\nVector store saved and reloaded successfully ✅")

In [ ]:
from langchain_community.vectorstores import Chroma

# ── Chroma Vector Store (persistent) ─────────────────────────
chroma_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="/tmp/chroma_db",
    collection_name="langchain_notes"
)

results = chroma_store.similarity_search("What is RAG?", k=2)
for r in results:
    print(r.page_content)

---
## 10. Retrieval & RAG

**Retrieval-Augmented Generation (RAG)** combines vector search with LLM generation.

In [ ]:
# ── Create a Retriever from VectorStore ──────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",   # options: similarity, mmr, similarity_score_threshold
    search_kwargs={"k": 3}      # return top 3 results
)

# Retrieve documents for a query
retrieved = retriever.invoke("Tell me about LLMs")
for doc in retrieved:
    print("-", doc.page_content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── Full RAG Chain ────────────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("What is retrieval augmented generation?")
print("RAG Answer:", answer)

In [ ]:
# ── RetrievalQA (legacy convenience chain) ───────────────────
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",          # stuff | map_reduce | refine | map_rerank
    retriever=retriever,
    return_source_documents=True,
    verbose=True
)

result = qa_chain.invoke({"query": "How does RAG improve LLMs?"})
print("Answer:", result["result"])
print("\nSources:")
for src in result["source_documents"]:
    print(" -", src.metadata)

---
## 11. Output Parsers

Parse and structure LLM outputs into Python objects.

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

# ── StrOutputParser ───────────────────────────────────────────
str_parser = StrOutputParser()
chain = ChatPromptTemplate.from_template("Name 3 programming languages.") | llm | str_parser
print("String output:", chain.invoke({}))

In [ ]:
# ── CommaSeparatedListOutputParser ───────────────────────────
list_parser = CommaSeparatedListOutputParser()

template = PromptTemplate(
    template="List 5 {item}. Return ONLY as comma-separated values.\n{format_instructions}",
    input_variables=["item"],
    partial_variables={"format_instructions": list_parser.get_format_instructions()}
)

chain = template | llm | list_parser
result = chain.invoke({"item": "Python libraries for data science"})
print("List result:", result)
print("Type:", type(result))

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# ── PydanticOutputParser ──────────────────────────────────────
class MovieReview(BaseModel):
    title: str = Field(description="Title of the movie")
    year: int = Field(description="Release year")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    pros: List[str] = Field(description="List of positives")
    cons: List[str] = Field(description="List of negatives")

pydantic_parser = PydanticOutputParser(pydantic_object=MovieReview)

prompt = ChatPromptTemplate.from_template(
    "Review the movie '{movie}' briefly.\n{format_instructions}"
)
prompt = prompt.partial(format_instructions=pydantic_parser.get_format_instructions())

chain = prompt | llm | pydantic_parser
review = chain.invoke({"movie": "Inception"})

print(f"Title: {review.title}")
print(f"Year: {review.year}")
print(f"Rating: {review.rating}/10")
print(f"Pros: {review.pros}")
print(f"Cons: {review.cons}")

In [ ]:
# ── JsonOutputParser ──────────────────────────────────────────
json_parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_template(
    "Provide info about {country} as JSON with keys: capital, population, continent, currency."
)

chain = prompt | llm | json_parser
data = chain.invoke({"country": "Germany"})
print(type(data), "\n", data)

---
## 12. Callbacks & Streaming

Monitor LangChain execution and stream tokens in real-time.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Token Streaming ───────────────────────────────────────────
streaming_llm = ChatOpenAI(model="gpt-4o-mini", streaming=True)

prompt = ChatPromptTemplate.from_template("Write a short poem about {topic}.")
chain = prompt | streaming_llm | StrOutputParser()

print("Streaming output:")
for chunk in chain.stream({"topic": "the ocean"}):
    print(chunk, end="", flush=True)
print()  # newline

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

# ── Custom Callback Handler ───────────────────────────────────
class LoggingCallback(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print(f"\n🔵 LLM started. Prompt: '{prompts[0][:60]}...'")
    
    def on_llm_end(self, response, **kwargs):
        text = response.generations[0][0].text
        print(f"✅ LLM finished. Output length: {len(text)} chars")
    
    def on_chain_start(self, serialized, inputs, **kwargs):
        print(f"⛓️ Chain started: {serialized.get('name', 'unknown')}")

    def on_chain_end(self, outputs, **kwargs):
        print("✅ Chain ended.")

callback = LoggingCallback()

llm_with_cb = ChatOpenAI(model="gpt-4o-mini", callbacks=[callback])
result = llm_with_cb.invoke("What is the capital of Australia?")
print("\nAnswer:", result.content)

---
## 13. LCEL — LangChain Expression Language

LCEL is the modern, composable way to build chains using the `|` (pipe) operator.

In [ ]:
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableParallel,
    RunnableBranch
)
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── RunnableLambda: wrap any function ─────────────────────────
shout = RunnableLambda(lambda x: x.upper())
print(shout.invoke("hello world"))

In [ ]:
# ── RunnableParallel: run multiple chains simultaneously ──────
prompt_pros = ChatPromptTemplate.from_template("List 3 pros of {thing}.")
prompt_cons = ChatPromptTemplate.from_template("List 3 cons of {thing}.")
prompt_summary = ChatPromptTemplate.from_template("Write one sentence summary about {thing}.")

analysis = RunnableParallel(
    pros    = prompt_pros    | llm | StrOutputParser(),
    cons    = prompt_cons    | llm | StrOutputParser(),
    summary = prompt_summary | llm | StrOutputParser()
)

result = analysis.invoke({"thing": "remote work"})
print("=== PROS ===")
print(result["pros"])
print("\n=== CONS ===")
print(result["cons"])
print("\n=== SUMMARY ===")
print(result["summary"])

In [ ]:
# ── RunnableBranch: conditional routing ──────────────────────
simple_prompt = ChatPromptTemplate.from_template("Answer simply: {question}")
detailed_prompt = ChatPromptTemplate.from_template("Answer in detail with examples: {question}")

branch = RunnableBranch(
    (lambda x: "simple" in x["question"].lower(), simple_prompt | llm | StrOutputParser()),
    (lambda x: "detail" in x["question"].lower(), detailed_prompt | llm | StrOutputParser()),
    simple_prompt | llm | StrOutputParser()  # default
)

print(branch.invoke({"question": "Simply explain what is AI?"}))
print("\n---")
print(branch.invoke({"question": "In detail, how does backpropagation work?"})[:200] + "...")

In [ ]:
# ── RunnablePassthrough: pass input unchanged ─────────────────
chain_with_input = RunnableParallel(
    original = RunnablePassthrough(),
    answer   = ChatPromptTemplate.from_template("Answer: {question}") | llm | StrOutputParser()
)

result = chain_with_input.invoke({"question": "What is 5 factorial?"})
print("Original input:", result["original"])
print("Answer:", result["answer"])

In [ ]:
# ── LCEL Runnable Methods ─────────────────────────────────────
chain = ChatPromptTemplate.from_template("Explain {topic} in one sentence.") | llm | StrOutputParser()

# .invoke() — single call
r = chain.invoke({"topic": "transformers"})
print("invoke:", r)

# .batch() — multiple inputs at once
results = chain.batch([
    {"topic": "LSTM"},
    {"topic": "attention mechanism"}
])
for r in results:
    print("batch:", r)

# .stream() — stream tokens
print("\nstream:", end=" ")
for chunk in chain.stream({"topic": "embeddings"}):
    print(chunk, end="", flush=True)
print()

# .ainvoke() / .astream() — async versions (use in async context)
# result = await chain.ainvoke({"topic": "vector databases"})

---
## 14. Structured Output & Tool Binding

The modern way to get structured outputs and bind tools to a model.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List, Optional

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── with_structured_output (recommended modern approach) ──────
class Person(BaseModel):
    name: str = Field(description="Full name of the person")
    age: Optional[int] = Field(description="Age if mentioned")
    occupation: str = Field(description="Job or role")
    skills: List[str] = Field(description="List of skills or expertise")

structured_llm = llm.with_structured_output(Person)

result = structured_llm.invoke(
    "Alice is a 32-year-old machine learning engineer who knows Python, PyTorch, and SQL."
)
print(type(result))
print(result)
print(f"Name: {result.name}, Age: {result.age}, Skills: {result.skills}")

In [ ]:
from langchain_core.tools import tool
import json

# ── bind_tools: attach tools to the model ────────────────────
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Simulated response
    return f"Weather in {city}: 22°C, partly cloudy"

@tool
def get_population(city: str) -> str:
    """Get the population of a city."""
    populations = {"paris": "2.1 million", "tokyo": "13.9 million", "delhi": "32 million"}
    return populations.get(city.lower(), "unknown")

llm_with_tools = llm.bind_tools([get_weather, get_population])

response = llm_with_tools.invoke("What's the weather in Paris and its population?")

print("Tool calls requested:")
for tc in response.tool_calls:
    print(f"  Tool: {tc['name']}, Args: {tc['args']}")

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

# ── Full tool-calling loop ────────────────────────────────────
tool_map = {"get_weather": get_weather, "get_population": get_population}

messages = [HumanMessage(content="What's the weather and population of Tokyo?")]
response = llm_with_tools.invoke(messages)
messages.append(response)

# Execute each tool call and append results
for tc in response.tool_calls:
    tool_fn = tool_map[tc["name"]]
    tool_result = tool_fn.invoke(tc["args"])
    messages.append(ToolMessage(content=tool_result, tool_call_id=tc["id"]))

# Final response with tool results incorporated
final = llm.invoke(messages)
print(final.content)

---
## 15. Advanced Retrievers

Beyond basic similarity search — smarter retrieval strategies.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="Python is great for data science.", metadata={"id": 1}),
    Document(page_content="Python is widely used in machine learning.", metadata={"id": 2}),
    Document(page_content="LangChain is a Python framework for LLMs.", metadata={"id": 3}),
    Document(page_content="JavaScript is used for web development.", metadata={"id": 4}),
    Document(page_content="Vector databases enable semantic search.", metadata={"id": 5}),
]
vs = FAISS.from_documents(docs, embeddings)

# ── MMR Retriever: balances relevance + diversity ─────────────
mmr_retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.5}  # 0=max diversity, 1=max relevance
)

results = mmr_retriever.invoke("Python for AI")
print("MMR Results (diverse):")
for r in results:
    print(f"  [{r.metadata['id']}] {r.page_content}")

In [ ]:
from langchain.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI

# ── MultiQueryRetriever: generates multiple query variants ────
# Generates 3 different phrasings of your query, retrieves for each, deduplicates
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=vs.as_retriever(search_kwargs={"k": 2}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)
)

import logging
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

results = mq_retriever.invoke("How is Python used in AI?")
print(f"\nMultiQuery returned {len(results)} unique documents:")
for r in results:
    print(f"  - {r.page_content}")

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# ── ContextualCompressionRetriever: extracts only relevant parts ──
compressor = LLMChainExtractor.from_llm(
    ChatOpenAI(model="gpt-4o-mini", temperature=0)
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vs.as_retriever(search_kwargs={"k": 3})
)

compressed_docs = compression_retriever.invoke("What is Python used for?")
print("Compressed results (only relevant parts kept):")
for d in compressed_docs:
    print(f"  - {d.page_content}")

In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# ── EnsembleRetriever: combine dense + sparse retrieval ───────
# BM25 = keyword-based (sparse), FAISS = semantic (dense)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2

dense_retriever = vs.as_retriever(search_kwargs={"k": 2})

ensemble = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6]  # BM25=40%, Dense=60%
)

results = ensemble.invoke("Python machine learning")
print(f"Ensemble returned {len(results)} docs:")
for r in results:
    print(f"  - {r.page_content}")

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# ── ParentDocumentRetriever: index small chunks, return full parent ──
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=0)
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)

vectorstore_pd = Chroma(collection_name="parent_docs", embedding_function=embeddings)
store = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore_pd,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

parent_retriever.add_documents(docs)
results = parent_retriever.invoke("Python AI")
print(f"Parent docs returned: {len(results)}")
for r in results:
    print(f"  - {r.page_content}")

---
## 16. Summarization Chains

Summarize long documents using different strategies.

In [ ]:
from langchain.chains.summarize import load_summarize_chain
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

docs = [
    Document(page_content="Artificial intelligence is transforming industries worldwide. From healthcare diagnostics to autonomous vehicles, AI is enabling machines to perform complex tasks that once required human intelligence."),
    Document(page_content="Machine learning, a subset of AI, allows systems to learn patterns from data without being explicitly programmed. Deep learning, using neural networks with multiple layers, has achieved breakthroughs in image recognition and natural language processing."),
    Document(page_content="Large language models like GPT-4 are trained on vast corpora of text. They can generate human-like text, answer questions, write code, and assist in creative tasks. Frameworks like LangChain simplify integrating these models into applications."),
]

# ── stuff: combine all docs into one prompt (best for small docs) ──
stuff_chain = load_summarize_chain(llm, chain_type="stuff", verbose=False)
result = stuff_chain.invoke({"input_documents": docs})
print("[STUFF] Summary:")
print(result["output_text"])

In [ ]:
# ── map_reduce: summarize each doc, then summarize summaries ──
map_reduce_chain = load_summarize_chain(llm, chain_type="map_reduce", verbose=False)
result = map_reduce_chain.invoke({"input_documents": docs})
print("[MAP_REDUCE] Summary:")
print(result["output_text"])

In [ ]:
# ── refine: iteratively refine the summary with each doc ─────
refine_chain = load_summarize_chain(llm, chain_type="refine", verbose=False)
result = refine_chain.invoke({"input_documents": docs})
print("[REFINE] Summary:")
print(result["output_text"])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Custom summarization with LCEL ───────────────────────────
summary_prompt = ChatPromptTemplate.from_template("""
Summarize the following text as bullet points. Be concise.

Text: {text}

Bullet-point summary:
""")

summarize_chain = summary_prompt | llm | StrOutputParser()

combined_text = "\n\n".join(d.page_content for d in docs)
summary = summarize_chain.invoke({"text": combined_text})
print("[CUSTOM LCEL] Summary:")
print(summary)

---
## 17. Caching

Cache LLM responses to save API costs and speed up repeated calls.

In [ ]:
import time
import langchain
from langchain.cache import InMemoryCache, SQLiteCache
from langchain_openai import ChatOpenAI

# ── InMemoryCache: fast, lives only in current session ───────
langchain.llm_cache = InMemoryCache()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

question = "What is the speed of light?"

t0 = time.time()
r1 = llm.invoke(question)
print(f"First call  ({time.time()-t0:.2f}s): {r1.content[:80]}")

t0 = time.time()
r2 = llm.invoke(question)  # served from cache
print(f"Second call ({time.time()-t0:.4f}s): {r2.content[:80]}")
print("\n✅ Second call was instant — served from cache!")

In [ ]:
# ── SQLiteCache: persists across Python sessions ─────────────
langchain.llm_cache = SQLiteCache(database_path="/tmp/langchain_cache.db")

llm_cached = ChatOpenAI(model="gpt-4o-mini", temperature=0)

r = llm_cached.invoke("What is Newton's first law?")
print("SQLite cached response:", r.content[:120])
print("\nRun again — it will be loaded from the SQLite database!")

In [ ]:
from langchain_community.cache import GPTCache

# ── Semantic Cache concept (requires gptcache library) ────────
# pip install gptcache
# Semantic caching matches similar (not exact) queries

# from gptcache import Cache
# from gptcache.manager.factory import manager_factory
# from gptcache.processor.pre import get_prompt
# 
# cache_obj = Cache()
# cache_obj.init(pre_embedding_func=get_prompt)
# langchain.llm_cache = GPTCache(cache_obj)
# 
# r1 = llm.invoke("What is ML?")          # API call
# r2 = llm.invoke("What is machine learning?")  # cache hit!

print("Semantic cache requires `pip install gptcache` — see comments above for setup.")

# Clear the cache
langchain.llm_cache = None
print("Cache cleared.")

---
## 18. Async Support

LangChain fully supports `async/await` for high-throughput, non-blocking applications.

In [ ]:
import asyncio
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = ChatPromptTemplate.from_template("Explain {topic} in one line.") | llm | StrOutputParser()

# ── ainvoke: single async call ────────────────────────────────
async def single_async_call():
    result = await chain.ainvoke({"topic": "neural networks"})
    print("ainvoke:", result)

await single_async_call()  # Use in Jupyter; in scripts: asyncio.run(single_async_call())

In [ ]:
import time

# ── abatch: run multiple calls concurrently ───────────────────
topics = ["transformers", "LSTM", "CNN", "GAN", "reinforcement learning"]

async def concurrent_calls():
    t0 = time.time()
    results = await chain.abatch(
        [{"topic": t} for t in topics],
        config={"max_concurrency": 5}
    )
    elapsed = time.time() - t0
    print(f"5 concurrent calls completed in {elapsed:.2f}s")
    for topic, result in zip(topics, results):
        print(f"  [{topic}]: {result}")

await concurrent_calls()

In [ ]:
# ── astream: async token streaming ───────────────────────────
async def stream_tokens():
    print("Streaming async: ", end="")
    async for chunk in chain.astream({"topic": "quantum computing"}):
        print(chunk, end="", flush=True)
    print()

await stream_tokens()

In [ ]:
# ── asyncio.gather for maximum parallelism ────────────────────
async def parallel_with_gather():
    tasks = [
        chain.ainvoke({"topic": topic})
        for topic in ["backpropagation", "attention", "dropout"]
    ]
    results = await asyncio.gather(*tasks)
    for topic, result in zip(["backpropagation", "attention", "dropout"], results):
        print(f"{topic}: {result}")

await parallel_with_gather()

---
## 19. Multi-modal (Vision)

Pass images to vision-capable models like GPT-4o.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

vision_llm = ChatOpenAI(model="gpt-4o", max_tokens=1024)

# ── Describe image from URL ───────────────────────────────────
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"

message = HumanMessage(content=[
    {"type": "text", "text": "Describe this image in detail."},
    {"type": "image_url", "image_url": {"url": image_url}}
])

response = vision_llm.invoke([message])
print(response.content)

In [ ]:
import base64
import httpx

# ── Describe image from base64 ────────────────────────────────
def image_url_to_base64(url: str) -> str:
    data = httpx.get(url).content
    return base64.standard_b64encode(data).decode("utf-8")

b64_image = image_url_to_base64(image_url)

message_b64 = HumanMessage(content=[
    {"type": "text", "text": "What's the main subject of this image?"},
    {
        "type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{b64_image}"}
    }
])

response = vision_llm.invoke([message_b64])
print(response.content)

In [ ]:
# ── Multiple images in one message ───────────────────────────
url1 = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/320px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
url2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg"

multi_image_msg = HumanMessage(content=[
    {"type": "text", "text": "Compare these two images briefly."},
    {"type": "image_url", "image_url": {"url": url1}},
    {"type": "image_url", "image_url": {"url": url2}},
])

response = vision_llm.invoke([multi_image_msg])
print(response.content)

---
## 20. LangSmith — Tracing & Debugging

LangSmith provides full observability: traces, evaluations, and dataset management.

In [ ]:
import os

# ── Setup: enable tracing via environment variables ───────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"          # Enable tracing
os.environ["LANGCHAIN_API_KEY"]     = "your-langsmith-api-key"
os.environ["LANGCHAIN_PROJECT"]     = "langchain-notebook"  # Project name in LangSmith
os.environ["LANGCHAIN_ENDPOINT"]    = "https://api.smith.langchain.com"

print("LangSmith tracing configured.")
print("All subsequent chain/agent calls will be logged to LangSmith automatically!")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# With tracing ON, every call is automatically traced
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = ChatPromptTemplate.from_template("Explain {topic} in one line.") | llm | StrOutputParser()

result = chain.invoke({"topic": "LangSmith"}, config={"run_name": "explain-topic-run"})
print(result)
print("\nView this trace at: https://smith.langchain.com")

In [ ]:
# ── Evaluation with LangSmith ─────────────────────────────────
# pip install langsmith
from langsmith import Client
from langsmith.evaluation import evaluate

# client = Client()

# # Create a dataset
# dataset = client.create_dataset("qa-test", description="QA test dataset")
# client.create_examples(
#     inputs  = [{"question": "What is AI?"}, {"question": "What is ML?"}],
#     outputs = [{"answer": "Simulation of human intelligence."}, {"answer": "Learning from data."}],
#     dataset_id=dataset.id
# )

# # Define evaluator
# def correctness_evaluator(run, example):
#     score = 1 if example.outputs["answer"].lower() in run.outputs["output"].lower() else 0
#     return {"key": "correctness", "score": score}

# # Run evaluation
# results = evaluate(
#     lambda inputs: chain.invoke(inputs),
#     data="qa-test",
#     evaluators=[correctness_evaluator],
# )

print("Uncomment above after setting up your LangSmith API key.")
print("Full docs: https://docs.smith.langchain.com")

---
## 21. LangGraph — Stateful Agent Workflows

LangGraph lets you build stateful, multi-actor agent workflows as graphs.

In [ ]:
!pip install langgraph -q

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# ── Define State ──────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]  # append-only list
    step: int

# ── Define Nodes ──────────────────────────────────────────────
def node_greet(state: AgentState) -> AgentState:
    print(f"  [node_greet] Step {state['step']}")
    return {"messages": ["Hello from the greet node!"], "step": state["step"] + 1}

def node_process(state: AgentState) -> AgentState:
    print(f"  [node_process] Step {state['step']}, messages so far: {len(state['messages'])}")
    return {"messages": ["Processing complete."], "step": state["step"] + 1}

def node_finalize(state: AgentState) -> AgentState:
    print(f"  [node_finalize] Step {state['step']}")
    return {"messages": ["Workflow done!"], "step": state["step"] + 1}

# ── Build the Graph ───────────────────────────────────────────
graph = StateGraph(AgentState)
graph.add_node("greet",    node_greet)
graph.add_node("process",  node_process)
graph.add_node("finalize", node_finalize)

graph.set_entry_point("greet")
graph.add_edge("greet",   "process")
graph.add_edge("process", "finalize")
graph.add_edge("finalize", END)

app = graph.compile()

# ── Run the graph ─────────────────────────────────────────────
result = app.invoke({"messages": ["Start"], "step": 0})
print("\nFinal messages:", result["messages"])
print("Total steps:", result["step"])

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated
import operator

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

tools = [add]
llm_with_tools = llm.bind_tools(tools)

# ── State ─────────────────────────────────────────────────────
class State(TypedDict):
    messages: Annotated[list, operator.add]

# ── Nodes ─────────────────────────────────────────────────────
def call_llm(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def call_tool(state: State):
    from langchain_core.messages import ToolMessage
    tool_map = {"add": add}
    results = []
    for tc in state["messages"][-1].tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return {"messages": results}

# ── Conditional routing ───────────────────────────────────────
def should_call_tool(state: State):
    last = state["messages"][-1]
    return "call_tool" if hasattr(last, "tool_calls") and last.tool_calls else END

# ── Build Graph ───────────────────────────────────────────────
workflow = StateGraph(State)
workflow.add_node("llm",  call_llm)
workflow.add_node("tool", call_tool)

workflow.set_entry_point("llm")
workflow.add_conditional_edges("llm", should_call_tool, {"call_tool": "tool", END: END})
workflow.add_edge("tool", "llm")  # after tool, go back to LLM

agent = workflow.compile()

result = agent.invoke({"messages": [HumanMessage(content="What is 42 + 58?")]})
print("Final response:", result["messages"][-1].content)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# ── Persistent State with Checkpointer ───────────────────────
# Enables conversation memory & time-travel debugging
memory = MemorySaver()
agent_with_memory = workflow.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "thread_001"}}

r1 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="What is 10 + 5?")]},
    config=config
)
print("Turn 1:", r1["messages"][-1].content)

r2 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="Now multiply that by 2.")]},
    config=config  # same thread_id = same conversation
)
print("Turn 2:", r2["messages"][-1].content)

---
## 🎯 Quick Reference Summary

| Module | Key Classes/Methods | Use Case |
|--------|-------------------|----------|
| **LLMs** | `ChatOpenAI`, `invoke()`, `batch()` | Interact with language models |
| **Prompts** | `ChatPromptTemplate`, `FewShotPromptTemplate` | Structure prompts |
| **Chains** | `\|` operator, `LLMChain` | Link components |
| **Memory** | `ConversationBufferMemory`, `RunnableWithMessageHistory` | Persist context |
| **Agents** | `create_react_agent`, `AgentExecutor`, `@tool` | Dynamic reasoning |
| **Loaders** | `PyPDFLoader`, `WebBaseLoader`, `CSVLoader` | Load documents |
| **Splitters** | `RecursiveCharacterTextSplitter` | Chunk documents |
| **Embeddings** | `OpenAIEmbeddings` | Convert text to vectors |
| **VectorStores** | `FAISS`, `Chroma` | Store & search vectors |
| **RAG** | `as_retriever()`, `RetrievalQA` | Grounded generation |
| **Parsers** | `PydanticOutputParser`, `JsonOutputParser` | Structured output |
| **LCEL** | `RunnableParallel`, `RunnableBranch`, `RunnableLambda` | Compose pipelines |
| **Structured Output** | `with_structured_output()`, `bind_tools()` | Type-safe LLM output |
| **Adv. Retrievers** | `MultiQueryRetriever`, `EnsembleRetriever`, `ParentDocumentRetriever` | Smarter retrieval |
| **Summarization** | `load_summarize_chain` (stuff/map_reduce/refine) | Summarize long docs |
| **Caching** | `InMemoryCache`, `SQLiteCache`, `GPTCache` | Save API costs |
| **Async** | `ainvoke()`, `abatch()`, `astream()` | Non-blocking execution |
| **Multi-modal** | Vision via `HumanMessage` + image_url | Image understanding |
| **LangSmith** | Tracing, datasets, evaluation | Observability & evals |
| **LangGraph** | `StateGraph`, `MemorySaver`, conditional edges | Stateful agent workflows |

---
### 📚 Resources
- [LangChain Docs](https://python.langchain.com)
- [LangChain GitHub](https://github.com/langchain-ai/langchain)
- [LangChain Hub](https://smith.langchain.com/hub)
- [LangSmith (Tracing)](https://smith.langchain.com)